In [ ]:
import os
import cv2
import numpy as np
import joblib
from deepface import DeepFace
from time import time

# --- 💡 LÍMITE DE IMÁGENES 💡 ---
# Pon 500 para una prueba decente, o 200 para una prueba súper rápida
MAX_IMAGES_PER_CLASS = 600 

print(f"Script 1: Extracción de Embeddings (MODO RÁPIDO: max {MAX_IMAGES_PER_CLASS} por clase)")

# --- Función LoadDataset (Adaptada de tu Deepface_kfold.ipynb) ---
def LoadDataset(folder, ext, max_per_class):
    nclasses = 0
    nperclass = []
    classlabels = []
    X = []
    Y = []

    print(f"Cargando dataset desde: {folder}")
    
    # Obtener lista de clases (directorios)
    class_list = [d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))]
    
    for class_name in class_list:
        class_folder = os.path.join(folder, class_name)
            
        nclasses += 1
        nsamples = 0
        print(f"\nCargando clase: {class_name} ({nclasses}/{len(class_list)})")

        for file_name in os.listdir(class_folder):
            
            # --- 💡 LÓGICA DEL LÍMITE 💡 ---
            if nsamples >= max_per_class:
                print(f"   ... Límite alcanzado ({max_per_class} imágenes)")
                break # Rompe el bucle de esta clase y pasa a la siguiente

            if file_name.endswith(ext):
                image_path = os.path.join(class_folder, file_name)
                try:
                    image = cv2.imread(image_path)
                    if image is None:
                        continue
                        
                    img1 = cv2.resize(image, dim, interpolation=cv2.INTER_AREA)

                    embedding_objs = DeepFace.represent(
                        img_path=img1,
                        model_name=model_name,
                        enforce_detection=False
                    )
                    img_embedding = embedding_objs[0]["embedding"]
                    
                    X.append(img_embedding)
                    Y.append(nclasses - 1)
                    nsamples += 1
                    
                    if nsamples % 100 == 0:
                        print(f"\r   ... procesadas {nsamples} imágenes", end="")
                
                except Exception as e:
                    # Ignora errores de 'represent' (ej. cara no encontrada)
                    pass

        print(f"\r   -> Clase '{class_name}' completada. Total: {nsamples} imágenes.")
        nperclass.append(nsamples)
        classlabels.append(class_name)

    X = np.array(X, dtype='float32')
    Y = np.array(Y, dtype='float64')

    if X.size == 0:
        return X, Y, 0, 0, 0, [], [], []

    n_samples, n_features = X.shape
    class_names = np.array(classlabels)
    n_classes = class_names.shape[0]

    return X, Y, n_samples, n_features, n_classes, classlabels, nperclass, class_names

# --- 1. Configuración del Modelo DeepFace ---
model_name = "Facenet"
print(f"Construyendo modelo: {model_name}")
model = DeepFace.build_model(model_name)
dim = (model.input_shape[1], model.input_shape[0]) 
print(f"Dimensiones de entrada: {dim}")

# --- 2. Carga del Dataset ---
folder = "C:/Users/lucia/Downloads/train" 

# ¡Ahora pasamos el límite como argumento!
X, Y, n_samples, n_features, n_classes, classlabels, nperclass, class_names = LoadDataset(folder, '.png', MAX_IMAGES_PER_CLASS)

print("\n--- Información del Dataset ---")
print(f"# Muestras: {n_samples}")
print(f"# Características (Embeddings): {n_features}")
print(f"# Clases: {n_classes}")

# --- 3. GUARDAR LOS DATOS EXTRAÍDOS ---
if n_samples > 0:
    print("\nGuardando datos extraídos en archivos .pkl...")
    
    joblib.dump(X, 'embeddings_X.pkl')
    joblib.dump(Y, 'labels_Y.pkl')
    joblib.dump(class_names, 'emotion_class_names.pkl')
    
    print("¡Éxito! Archivos 'embeddings_X.pkl', 'labels_Y.pkl' y 'emotion_class_names.pkl' guardados.")
    print("Ahora puedes ejecutar '2_entrenar_svm.py'.")
else:
    print("Error: No se cargaron muestras. Verifica la ruta de tu dataset ('folder') y la extensión ('.png').")

In [6]:
import numpy as np
import joblib
from time import time
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

print("Script 2: Entrenamiento del Modelo SVM (Rápido)")

# --- 1. Cargar datos pre-extraídos ---
try:
    print("Cargando 'embeddings_X.pkl' y 'labels_Y.pkl'...")
    X = joblib.load('embeddings_X.pkl')
    Y = joblib.load('labels_Y.pkl')
    print(f"Datos cargados: {X.shape[0]} muestras, {X.shape[1]} características.")
except FileNotFoundError:
    print("Error: No se encontraron los archivos .pkl.")
    print("Por favor, ejecuta '1_extraer_embeddings.py' primero.")
    exit()

# --- 2. Entrenamiento del Modelo SVM ---
if X.shape[0] > 0:
    print("\nEntrenando Scaler (MinMaxScaler)...")
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    print("Iniciando GridSearchCV para SVM...")
    t0 = time()
    parameters = {'C': [1e3, 5e3, 1e4], 
                  'gamma': [0.0001, 0.001, 0.01]} 
    
    clf = GridSearchCV(
        SVC(kernel='rbf', class_weight='balanced', probability=True), 
        parameters, 
        cv=3,
        n_jobs=-1, # Usar todos los cores (ahora irá rápido)
        verbose=3  # Imprime el progreso
    )
    clf.fit(X_scaled, Y)
    
    print(f"GridSearchCV terminado en {time() - t0:.3f}s")
    print("Mejor estimador encontrado:")
    print(clf.best_estimator_)
    
    # 2.3 Guardar los 2 modelos finales para el prototipo
    final_model = clf.best_estimator_
    print("\nGuardando modelos finales...")
    
    joblib.dump(final_model, 'emotion_svm_model.pkl')
    joblib.dump(scaler, 'emotion_scaler.pkl')
    
    print("¡Éxito! Archivos 'emotion_svm_model.pkl' y 'emotion_scaler.pkl' guardados.")
    print("¡Todo listo para ejecutar el prototipo!")
else:
    print("Error: Los datos cargados están vacíos.")

Script 2: Entrenamiento del Modelo SVM (Rápido)
Cargando 'embeddings_X.pkl' y 'labels_Y.pkl'...
Datos cargados: 3436 muestras, 128 características.

Entrenando Scaler (MinMaxScaler)...
Iniciando GridSearchCV para SVM...
Fitting 3 folds for each of 9 candidates, totalling 27 fits
GridSearchCV terminado en 40.637s
Mejor estimador encontrado:
SVC(C=1000.0, class_weight='balanced', gamma=0.01, probability=True)

Guardando modelos finales...
¡Éxito! Archivos 'emotion_svm_model.pkl' y 'emotion_scaler.pkl' guardados.
¡Todo listo para ejecutar el prototipo!


TODO

In [8]:
import cv2
import joblib
import numpy as np
from deepface import DeepFace
import sys
import time 

print("Iniciando Prototipo 1: Detector de Emociones (Pulsa ESC para salir)")

# --- 1. Cargar nuestros modelos entrenados ---
try:
    svm_model = joblib.load('emotion_svm_model.pkl')
    scaler = joblib.load('emotion_scaler.pkl')
    class_names = joblib.load('emotion_class_names.pkl')
    print("Modelos cargados correctamente.")
    print(f"Clases detectadas: {class_names}")
except FileNotFoundError:
    print("Error: No se encontraron los archivos .pkl.")
    sys.exit()
except Exception as e:
    print(f"Error al cargar modelos: {e}")
    sys.exit()

# --- 2. Configurar DeepFace y Webcam ---
face_detector_backend = "mtcnn" 
embedding_model_name = "Facenet"
font = cv2.FONT_HERSHEY_SIMPLEX
color_reaccion = (0, 255, 0) 

print("Abriendo cámara...")
cap = cv2.VideoCapture(0) 
if not cap.isOpened():
    print("Cámara 0 no funciona, probando Cámara 1...")
    cap = cv2.VideoCapture(1) 
    if not cap.isOpened():
        print("Error: No se puede abrir ninguna cámara.")
        sys.exit()

print("Cámara abierta. Esperando 1 segundo...")
time.sleep(1) 

# --- 3. Bucle Principal (en tiempo real) ---
while True:
    ret, frame = cap.read()
    if not ret:
        print("Error al capturar frame. ¿Cámara desconectada?")
        break

    try:
        # 3.1 Detectar caras en el frame
        faces = DeepFace.extract_faces(
            img_path=frame, 
            detector_backend=face_detector_backend, 
            enforce_detection=False
        )

        # 3.2 Iterar sobre cada cara encontrada
        for face in faces:
            # --- ¡LÍNEA ERRÓNEA ELIMINADA! ---
            # Ya no comprobamos la 'confidence'

            x, y, w, h = face['facial_area']['x'], face['facial_area']['y'], face['facial_area']['w'], face['facial_area']['h']
            face_img = frame[y:y+h, x:x+w]
            
            if face_img.size == 0:
                continue

            try:
                # 3.3 Obtener el embedding de la cara
                embedding_obj = DeepFace.represent(
                    img_path=face_img, 
                    model_name=embedding_model_name, 
                    enforce_detection=False 
                )
                embedding = embedding_obj[0]["embedding"]

                # 3.4 Predecir con NUESTRO modelo SVM
                embedding_np = np.array(embedding).reshape(1, -1)
                embedding_scaled = scaler.transform(embedding_np)
                prediction_index = svm_model.predict(embedding_scaled)
                emotion_label = class_names[prediction_index[0]]

                # 3.5 La "Reacción": Dibujar en pantalla
                cv2.rectangle(frame, (x, y), (x + w, y + h), color_reaccion, 2)
                cv2.putText(frame, emotion_label, (x, y - 10), font, 0.9, color_reaccion, 2, cv2.LINE_AA)

            except Exception as e:
                pass 

    except Exception as e:
        pass 

    # 3.6 Mostrar el fotograma final
    cv2.imshow('Prototipo 1 - Detector de Emociones (Pulsa ESC para salir)', frame)

    # Salir con la tecla ESC
    if cv2.waitKey(1) & 0xFF == 27:
        break

# --- 4. Limpieza ---
print("Cerrando...")
cap.release()
cv2.destroyAllWindows()

Iniciando Prototipo 1: Detector de Emociones (Pulsa ESC para salir)
Modelos cargados correctamente.
Clases detectadas: ['angry' 'disgusted' 'fearful' 'happy' 'neutral' 'sad' 'surprised']
Abriendo cámara...
Cámara abierta. Esperando 1 segundo...
Cerrando...


ENTRENAMIENTO 2

In [1]:
# Entrenamiento del modelo SVM con tus embeddings

import joblib
import numpy as np
from time import time
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

print("=== Entrenamiento del Modelo SVM ===")

# --- 1. Cargar los datos ya extraídos ---
try:
    X = joblib.load('embeddings_X.pkl')
    Y = joblib.load('labels_Y.pkl')
    print(f"Datos cargados: {X.shape[0]} muestras, {X.shape[1]} características")
except FileNotFoundError:
    print("No se encontraron los archivos embeddings_X.pkl / labels_Y.pkl")
    raise

# --- 2. Escalado de características ---
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
print("Escalado de características completado.")

# --- 3. GridSearch para encontrar los mejores hiperparámetros ---
parameters = {'C': [1e3, 5e3, 1e4], 'gamma': [0.0001, 0.001, 0.01]}
clf = GridSearchCV(
    SVC(kernel='rbf', class_weight='balanced', probability=True),
    parameters,
    cv=3,
    n_jobs=-1,
    verbose=3
)

print("Iniciando GridSearchCV...")
t0 = time()
clf.fit(X_scaled, Y)
print(f"GridSearchCV completado en {time() - t0:.2f} segundos")
print("Mejor estimador encontrado:")
print(clf.best_estimator_)

# --- 4. Guardar modelo y scaler ---
joblib.dump(clf.best_estimator_, 'emotion_svm_model.pkl')
joblib.dump(scaler, 'emotion_scaler.pkl')
print("Modelos guardados: 'emotion_svm_model.pkl' y 'emotion_scaler.pkl'")


=== Entrenamiento del Modelo SVM ===
Datos cargados: 3436 muestras, 128 características
Escalado de características completado.
Iniciando GridSearchCV...
Fitting 3 folds for each of 9 candidates, totalling 27 fits
GridSearchCV completado en 77.68 segundos
Mejor estimador encontrado:
SVC(C=1000.0, class_weight='balanced', gamma=0.01, probability=True)
Modelos guardados: 'emotion_svm_model.pkl' y 'emotion_scaler.pkl'
